In [4]:
using Symbolics, Nemo, Groebner
using StaticArrays
using Latexify
using Flight

In [ ]:
# @variables t
parameter_vars = @variables g m1 m2 L R J1 J2 k_motor b_motor J_motor
input_vars = @variables u
state_vars = @variables ω1 ω2 v θ η
dstate_vars = @variables ω̇1 ω̇2 v̇ θ̇ η̇
internal_vars = @variables τ_m F_21x F_21z F_i2x F_i2z

J1 = 1/12 * m1 * (2L)^2
J_wheel = 1/2 * m2 * R^2
J2 = J_motor + J_wheel

#equations
eq_noslip = v̇ - R*ω̇2 ~ 0
eq_motor = τ_m ~ k_motor * u - b_motor * (ω2 - ω1)
eqs_dynamics1 = [
    F_21x ~ m1 * v̇ + m1 * L * (ω̇1 * cos(θ) - ω1^2 * sin(θ)),
    F_21z - m1*g ~ m1 * L * (-ω̇1 * sin(θ) - ω1^2 * cos(θ)),
    F_i2x - F_21x ~ m2 * v̇,
    F_i2z - F_21z - m2 * g ~ 0,
    -τ_m + L * (F_21z * sin(θ) - F_21x * cos(θ)) ~ J1 * ω̇1,
    τ_m - R * F_i2x ~ J2 * ω̇2
]
eqs_dynamics2 = [
    m1*L * cos(θ) * ω̇1 + m1 * R * ω̇2 - F_21x ~ m1 * L * ω1^2 * sin(θ),
    m1*L * sin(θ) * ω̇1 + F_21z ~ m1 * g - m1*L * ω1^2 * cos(θ),
    J1 * ω̇1 + L * cos(θ) * F_21x - L * sin(θ) * F_21z + τ_m ~ 0,
    m2 * R * ω̇2 + F_21x - F_i2x ~ 0,
    -F_21z + F_i2z ~ m2 * g,
    J2 * ω̇2 + R * F_i2x - τ_m ~ 0
]
eqs_kinematics = [
    θ̇ ~ ω1
    η̇ ~ v
]


#the linear and angular momentum equations, no-slip rolling condition for the
#wheel and motor model give:
coupled_eqs1 = [eqs_dynamics1..., eq_noslip, eq_motor]
coupled_eqs2 = [eqs_dynamics2..., eq_noslip, eq_motor]

parameter_vals = Dict(
    g => 9.81,
    m1 => 0.5,
    m2 => 0.1,
    L => 0.15,
    R => 0.05,
    k_motor => 0.32,
    b_motor => 0.0189,
    J_motor => 0.0014
)

input_vals = Dict(
    u => 0.23,
)

state_vals = Dict(
    ω1 => 0.05,
    ω2 => 0.4,
    θ => π/6,
)

all_vals = merge(parameter_vals, input_vals, state_vals)

solvable_eqs1 = substitute(coupled_eqs1, all_vals)
solvable_eqs2 = substitute(coupled_eqs2, all_vals)

latexify(solvable_eqs1)
latexify(solvable_eqs2)


"\\begin{align}\n - \\mathtt{F\\_21x} + 0.064952 \\mathtt{{\\dot{\\omega}}1} + 0.025 \\mathtt{{\\dot{\\omega}}2} &= 9.375 \\cdot 10^{-5} \\\\\n\\mathtt{F\\_21z} + 0.0375 \\mathtt{{\\dot{\\omega}}1} &= 4.9048 \\\\\n0.1299 \\mathtt{F\\_21x} - 0.075 \\mathtt{F\\_21z} + \\mathtt{\\tau\\_m} + 0.0009375 \\m" ⋯ 66 bytes ⋯ " 0.005 \\mathtt{{\\dot{\\omega}}2} &= 0 \\\\\n - \\mathtt{F\\_21z} + \\mathtt{F\\_i2z} &= 0.981 \\\\\n0.05 \\mathtt{F\\_i2x} - \\mathtt{\\tau\\_m} + 0.001525 \\mathtt{{\\dot{\\omega}}2} &= 0 \\\\\n\\mathtt{\\dot{v}} - 0.05 \\mathtt{{\\dot{\\omega}}2} &= 0 \\\\\n\\mathtt{\\tau\\_m} &= 0.066985\n\\end{align}\n"

In [6]:
solved_eqs1 = symbolic_linear_solve(solvable_eqs1, [ω̇1, ω̇2, v̇, F_21x, F_21z, F_i2x, F_i2z, τ_m])

8-element Vector{Float64}:
 26.315677601674434
 -6.106771058906435
 -0.3053385529453245
  1.5564903725911787
  3.9179997101739987
  1.5259565172966463
  4.898999710173999
  0.066985

In [5]:
solved_eqs2 = symbolic_linear_solve(solvable_eqs2, [ω̇1, ω̇2, v̇, F_21x, F_21z, F_i2x, F_i2z, τ_m])

8-element Vector{Float64}:
 19.887285293862963
  0.7946558552140596
  0.039732792760705404
  1.3114897171399018
  4.15906442171693
  1.3154629964159725
  5.14006442171693
  0.066985

In [6]:
A = [
    m1*L/2*cos(θ)   m1*R        -1          0           0           0           0
    m1*L/2*sin(θ)   0           0           1           0           0           0
    J1              0           L/2*cos(θ)  -L/2*sin(θ) 0           0           1
    0               m2*R        1           0           -1          0           0
    0               0           0           -1          0           1           0
    0               J2          0           0           R           0           -1
    0               0           0           0           0           0           1
]

b = [
    m1*L/2*ω1^2*sin(θ),
    m1*g - m1*L/2*ω1^2*cos(θ),
    0,
    0,
    m2*g,
    0,
    k_motor * u - b_motor * (ω2 - ω1)
]

A_solvable = Float64.(substitute(A, all_vals))
b_solvable = Float64.(substitute(b, all_vals))
solved_eqs3 = A_solvable\b_solvable

7-element Vector{Float64}:
 19.88728529386299
  0.7946558552139995
  1.311489717139902
  4.159064421716929
  1.3154629964159716
  5.140064421716929
  0.066985

In [ ]:
function num_solve()

    g = 9.81
    m1 = 0.5
    m2 = 0.1
    L = 0.3
    R = 0.05
    k_motor = 0.32
    b_motor = 0.0189
    J_motor = 0.0014
    J_wheel = 1/2 * m2 * R^2
    J1 = 1/12 * m1 * L^2
    J2 = J_motor + J_wheel

    u = 0.118125
    ω1 = 0.0
    ω2 = 2.0
    θ = 0

    #with a nonzero theta, we need to accelerate to keep ω̇1=0
    # u = 0.12839
    # ω1 = 0.0
    # ω2 = 2.0
    # θ = 0.01

    A = @SMatrix[
        m1*L/2*cos(θ)   m1*R        -1          0           0           0           0
        m1*L/2*sin(θ)   0           0           1           0           0           0
        J1              0           L/2*cos(θ)  -L/2*sin(θ) 0           0           1
        0               m2*R        1           0           -1          0           0
        0               0           0           -1          0           1           0
        0               J2          0           0           R           0           -1
        0               0           0           0           0           0           1
    ]

    b = @SVector[
        m1*L/2*ω1^2*sin(θ),
        m1*g - m1*L/2*ω1^2*cos(θ),
        0,
        0,
        m2*g,
        0,
        k_motor * u - b_motor * (ω2 - ω1)
    ]

    return A\b

end

num_solve (generic function with 1 method)

In [119]:
num_solve()

7-element SVector{7, Float64} with indices SOneTo(7):
 6.905737288607124e-5
 1.0857986934875474
 0.02715014638119215
 4.904999948207834
 0.03257913984862989
 5.885999948207833
 0.0032848000000000044